# Week 2: Image Preprocessing + Testing Existing OCR Tools

This notebook prepares the synthetic Urdu text images collected in Week 1
by cleaning and standardising them (preprocessing), then tests an existing
OCR tool (Tesseract) to see how well it performs on Urdu script.
The goal is to identify the gaps in existing OCR tools that this project
aims to solve.

In [ ]:
!pip install opencv-python-headless pillow

import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [20]:
def preprocess_image(image_path, save_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return

    # Step 1: Convert to grayscale (removes colour noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize to standard size (keeps all images same dimensions)
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Remove noise (makes text cleaner)
    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    # Step 4: Binarise (make pixels either pure black or pure white)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    # Save processed image
    cv2.imwrite(save_path, binary)
    return binary

# Create output folder (in Google Drive this time)
os.makedirs('/content/drive/MyDrive/data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [21]:
import glob

# Find all images in Google Drive data/raw/
all_images = glob.glob('/content/drive/MyDrive/data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('/content/drive/MyDrive/data/raw/**/*.png', recursive=True)

print(f'Found {len(all_images)} images to process')

processed_count = 0
for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'/content/drive/MyDrive/data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder in Google Drive')

Found 20 images to process
Done! Processed 20 images
Check data/processed/ folder in Google Drive


In [12]:
!find /content/drive/MyDrive -iname "*.png" 2>/dev/null | head -20

In [13]:
!find /content/drive/MyDrive -iname "*.png" 2>/dev/null | head -20

In [14]:
import os
print(os.path.exists('/content/drive/MyDrive'))
print(os.listdir('/content/drive/MyDrive'))

False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

In [15]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MessageError: Error: credential propagation was unsuccessful

In [16]:
import os
print(os.path.exists('/content/drive/MyDrive'))
if os.path.exists('/content/drive/MyDrive'):
    print(os.listdir('/content/drive/MyDrive'))

True
['Classroom', 'Screenshot_20260620_175057_NayaPay.jpg', 'Screenshot_20260620_175108_NayaPay.jpg', 'Colab Notebooks', 'week1_ML_SI-26.gdoc']


In [17]:
print(os.listdir('/content/drive/MyDrive'))

['Classroom', 'Screenshot_20260620_175057_NayaPay.jpg', 'Screenshot_20260620_175108_NayaPay.jpg', 'Colab Notebooks', 'week1_ML_SI-26.gdoc']


In [18]:
import os

for folder in ['Colab Notebooks', 'Classroom']:
    path = f'/content/drive/MyDrive/{folder}'
    print(f'--- {folder} ---')
    if os.path.exists(path):
        print(os.listdir(path))
    else:
        print('Does not exist')
    print()

--- Colab Notebooks ---
['SI26-Week1-fatima.ipynb..ipynb', 'SI26-Week2-Fatima.ipynb']

--- Classroom ---
['AI project C', 'Machine Learning | SI-26']



In [19]:
import os

for folder in ['Classroom/AI project C', 'Classroom/Machine Learning | SI-26']:
    path = f'/content/drive/MyDrive/{folder}'
    print(f'--- {folder} ---')
    if os.path.exists(path):
        print(os.listdir(path))
    else:
        print('Does not exist')
    print()

--- Classroom/AI project C ---
[]

--- Classroom/Machine Learning | SI-26 ---
['fatima sajid internship 1 task.pdf']



In [22]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
from PIL import Image

# Test on 5 of your processed images
test_images = list(glob.glob('/content/drive/MyDrive/data/processed/*.png'))[:5]

print('=== Tesseract Results on Urdu Images ===')
print()

for img_path in test_images:
    img = Image.open(img_path)
    # 'urd' tells Tesseract to use the Urdu language model
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {img_path}')
    print(f'Tesseract output: {result}')
    print('---')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 0s (6,207 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
=== Tesseract Results on Urdu Images ===

Image: /content/drive/MyDrive/data/processed/urdu_1.png
Tesseract output: س×صہ-

In [25]:
urdu_texts = [
    "اردو ایک خوبصورت زبان ہے",
    "پاکستان ہمارا وطن ہے",
    "یہ ایک مصنوعی ڈیٹاسیٹ کا نمونہ ہے",
    "میں کمپیوٹر سائنس کی طالبہ ہوں",
    "کتاب پڑھنا ایک اچھی عادت ہے",
    "سورج مشرق سے طلوع ہوتا ہے",
    "علم حاصل کرنا ہر مسلمان پر فرض ہے",
    "دوستی ایک انمول رشتہ ہے",
    "صبر کا پھل میٹھا ہوتا ہے",
    "محنت میں عظمت ہے",
    "وقت کی قدر کرنی چاہیے",
    "خوش باشتنی سے بڑھتی ہے",
    "سچائی ہمیشہ کامیاب ہوتی ہے",
    "درخت لگانا ایک نیک عمل ہے",
    "پانی زندگی کی نہایت ضروری ہے",
    "اساتذہ کا احترام کرنا چاہیے",
    "صحت ہی اصل دولت ہے",
    "نظم و ضبط کامیابی کی کنجی ہے",
    "خواب دیکھنا اور پورا کرنا",
    "امید پر دنیا قائم ہے"
]

In [26]:
import os

for img_path in test_images:
    filename = os.path.basename(img_path)
    num = int(filename.replace('urdu_', '').replace('.png', ''))
    actual_text = urdu_texts[num - 1]
    print(f'{filename} -> Actual: {actual_text}')

urdu_1.png -> Actual: اردو ایک خوبصورت زبان ہے
urdu_15.png -> Actual: پانی زندگی کی نہایت ضروری ہے
urdu_6.png -> Actual: سورج مشرق سے طلوع ہوتا ہے
urdu_14.png -> Actual: درخت لگانا ایک نیک عمل ہے
urdu_16.png -> Actual: اساتذہ کا احترام کرنا چاہیے


## Gap Analysis: Tesseract on Urdu OCR

| Image | Actual Text | Tesseract Output | Correct? |
|-------|------------|-------------------|----------|
| urdu_1.png | اردو ایک خوبصورت زبان ہے | garbled dots/symbols | ❌ |
| urdu_15.png | پانی زندگی کی نہایت ضروری ہے | "27" + random characters | ❌ |
| urdu_6.png | سورج مشرق سے طلوع ہوتا ہے | single dot/mark | ❌ |
| urdu_14.png | درخت لگانا ایک نیک عمل ہے | ".8" random symbols | ❌ |
| urdu_16.png | اساتذہ کا احترام کرنا چاہیے | "xx" + scrambled characters | ❌ |

**Result: 0 out of 5 images read correctly.**

Tesseract fails on Urdu because Urdu uses a cursive Nastaliq script where
letters connect and change shape depending on their position in a word —
very different from the separated Latin letters Tesseract was mainly
designed for. Urdu is also written right-to-left, adding another layer
of complexity. In our test of 5 preprocessed images, Tesseract correctly
read 0 out of 5 sentences, producing garbled symbols, random numbers, or
mostly empty output instead of real Urdu words. This shows that
off-the-shelf OCR tools are not reliable for Urdu text, which is exactly
the gap our project aims to address by building a custom Urdu OCR model.
